<a href="https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w08_warehouse_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — Warehouse Features and Time-Aware Labels

My Week 5 model was trained on the 30,000-row anonymized starter CSV. Two things
about that bothered me enough to redo it:

1. The features are **contemporaneous** with the label. The starter's 90-day
   aggregates cover the same window the decline label is measured over, so the
   result is an association, not a forecast.
2. The capstone card points at the **full warehouse**, and my Week 4 data
   contract already queries it with DuckDB over `hf://`. There is no good reason
   to model on the small slice.

This notebook rebuilds the feature table from
`fact_content_daily_performance` and defines a label on a **future window**:
features come from days before an anchor date T, the outcome is measured strictly
after T.

**Section 1** surveys what the warehouse actually contains, so the windows are
chosen from the data rather than assumed.

In [ ]:
# ============================================================
# CAPSTONE — SECTION 1
# WHAT IS ACTUALLY IN THE WAREHOUSE
# ============================================================

# REASONING:
# Before choosing a feature window and an outcome window I need to know how many
# months exist, how dense each one is, and whether coverage is stable enough to
# support a future-window label.
#
# This cell reads only aggregates. No row-level data and no private fields.

%pip install -q duckdb huggingface_hub pandas

import os

import duckdb
import pandas as pd

# ------------------------------------------------------------
# TOKEN
# ------------------------------------------------------------
#
# The token is never written into this notebook. In Colab it comes from
# Secrets; locally it comes from the HF_TOKEN environment variable.

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("Token source: Colab Secrets")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("Token source: HF_TOKEN environment variable")

if not HF_TOKEN:
    raise RuntimeError(
        "No Hugging Face token found.\n"
        "In Colab: add HF_TOKEN under the key icon in the left sidebar.\n"
        "Locally:  set HF_TOKEN in your environment before starting Jupyter."
    )

# ------------------------------------------------------------
# CONNECT
# ------------------------------------------------------------

con = duckdb.connect()

con.execute(
    "CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

daily = (
    f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet', "
    "hive_partitioning = true)"
)

print("Connected.\n")

# ------------------------------------------------------------
# MONTH-BY-MONTH COVERAGE
# ------------------------------------------------------------

coverage = con.execute(f"""
SELECT
    month,
    COUNT(*)                            AS rows,
    COUNT(DISTINCT client_hash_id)      AS clients,
    COUNT(DISTINCT content_hash_id)     AS pages,
    MIN(report_date)                    AS min_date,
    MAX(report_date)                    AS max_date,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_rows
FROM {daily}
GROUP BY month
ORDER BY month
""").df()

print("WAREHOUSE COVERAGE BY MONTH")
print("=" * 78)
print(coverage.to_string(index=False))

print("\nMonths:", len(coverage))
print("Total rows:", f"{coverage['rows'].sum():,}")
print("Date span:", coverage["min_date"].min(), "to", coverage["max_date"].max())


### What I am looking for in the output above

- **How many months**, and whether the row counts are steady or ramping. A
  ramping panel means early months have thinner client coverage.
- **Whether `pages` is stable** across months. If the page population churns
  heavily, a page present in the feature window may be absent from the outcome
  window, and those rows need an explicit decision rather than a silent drop.
- **The GSC vs GA4 split.** Week 4 found GA4 available on only 413,966 of
  9,841,378 March rows, so engagement features will be sparse and must be
  imputed honestly rather than zero-filled.

The feature window and outcome window are chosen in Section 2, once these
numbers are on the page.